# Flexible Sphere Packing with Polydispersity

This notebook demonstrates the new flexible packing capabilities:
1. **Zero Gravity Default**: Periodic packing without floor bias.
2. **Custom Initialization**: Setting random positions, velocities, and scales from Python.
3. **Polydispersity**: Particles have different base sizes.
4. **Growth Protocol**: `global_scale_factor` increases from 0 to 1 during the simulation.

In [ ]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

sys.path.append(os.path.abspath('../build'))
import demgpu

## 1. Initialization

We create 1000 spheres with random radii (0.4 to 0.6) and random initial velocities.

In [ ]:
NUM_PARTICLES = 1000
DOMAIN_SIZE = 10.0

sim = demgpu.Simulation(NUM_PARTICLES)
sim.initialize(shape_type=0) # Allocate

# 1. Random Scales (Polydispersity)
# Base Radius = 0.5. We scale this by 0.8 to 1.2 -> Radius [0.4, 0.6]
scales = np.random.uniform(0.8, 1.2, NUM_PARTICLES).astype(np.float32)
sim.set_scales(scales)

# 2. Random Positions
positions = np.random.uniform(0, DOMAIN_SIZE, (NUM_PARTICLES, 3)).astype(np.float32)
# Need to handle 4th component (inv_mass) if set_positions expects it
# The C++ binding handles (N,3) by defaulting mass, so this is fine.
sim.set_positions(positions)

# 3. Random Velocities
velocities = np.random.uniform(-1.0, 1.0, (NUM_PARTICLES, 3)).astype(np.float32)
sim.set_velocities(velocities)

# 4. Set Gravity to Zero (Default is 0, but explicit check)
sim.set_gravity(0.0, 0.0, 0.0) 

# 5. Start with Scale 0
sim.set_global_scale(0.0)

## 2. Growth Protocol

We linearly increase the `global_scale` from 0.0 to 1.0 over `T_GROWTH` seconds.

In [ ]:
T_GROWTH = 2.0  # seconds
DT = 0.005      # time step
STEPS = int(T_GROWTH / DT)

print(f"Starting growth protocol: {STEPS} steps...")
t0 = time.time()

for step in range(STEPS):
    t = step * DT
    current_scale = min(1.0, t / T_GROWTH)
    
    sim.set_global_scale(current_scale)
    sim.step(DT)

print(f"Growth complete in {time.time() - t0:.3f}s")

# Optional: Thermalize / Relax at full scale
print("Relaxing...")
sim.set_global_scale(1.0)
for _ in range(100):
    sim.step(DT)

## 3. Analysis

Visualize the polydisperse packing.

In [ ]:
pos_data = sim.get_positions().reshape(-1, 4)
pos = pos_data[:, :3]
final_scales = sim.get_scales()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with size correlated to scale
# s argument in scatter is area, so proportional to scale^2
sizes = 50 * (final_scales ** 2)

sc = ax.scatter(pos[:,0], pos[:,2], pos[:,1], s=sizes, c=final_scales, cmap='coolwarm', alpha=0.8)

ax.set_title("Polydisperse Sphere Packing")
plt.colorbar(sc, label='Scale Factor')
plt.show()